In [4]:
#2.将高德地图提供的坐标（GCJ-02 坐标系）转换为 WGS-84 坐标系，并且实现了不同坐标系之间的相互转换。

from ctypes import pointer
import math
import pandas as pd


In [5]:
x_pi = 3.14159265358979324 * 3000.0 / 180.0
pi = 3.1415926535897932384626  # π
a = 6378245.0  # 地球长半轴（单位：米）
ee = 0.00669342162296594323  # 地球椭球的偏心率平方

In [6]:
'''将 高德/火星GCJ-02 坐标系转换为 通用WGS-84 坐标系。'''
def gcj02towgs84(lng, lat):
    """
    GCJ02(火星坐标系)转GPS84
    :param lng: 火星坐标系的经度
    :param lat: 火星坐标系的纬度
    :return: 转换后的WGS84坐标
    """
    if out_of_china(lng, lat):
        return lng, lat
    dlat = transformlat(lng - 105.0, lat - 35.0)
    dlng = transformlng(lng - 105.0, lat - 35.0)
    radlat = lat / 180.0 * pi
    magic = math.sin(radlat)
    magic = 1 - ee * magic * magic
    sqrtmagic = math.sqrt(magic)
    dlat = (dlat * 180.0) / ((a * (1 - ee)) / (magic * sqrtmagic) * pi)
    dlng = (dlng * 180.0) / (a / sqrtmagic * math.cos(radlat) * pi)
    mglat = lat + dlat
    mglng = lng + dlng
    return [lng * 2 - mglng, lat * 2 - mglat]


def transformlat(lng, lat):
    ret = -100.0 + 2.0 * lng + 3.0 * lat + 0.2 * lat * lat + \
        0.1 * lng * lat + 0.2 * math.sqrt(math.fabs(lng))
    ret += (20.0 * math.sin(6.0 * lng * pi) + 20.0 *
            math.sin(2.0 * lng * pi)) * 2.0 / 3.0
    ret += (20.0 * math.sin(lat * pi) + 40.0 *
            math.sin(lat / 3.0 * pi)) * 2.0 / 3.0
    ret += (160.0 * math.sin(lat / 12.0 * pi) + 320 *
            math.sin(lat * pi / 30.0)) * 2.0 / 3.0
    return ret
def transformlng(lng, lat):
    ret = 300.0 + lng + 2.0 * lat + 0.1 * lng * lng + \
        0.1 * lng * lat + 0.1 * math.sqrt(math.fabs(lng))
    ret += (20.0 * math.sin(6.0 * lng * pi) + 20.0 *
            math.sin(2.0 * lng * pi)) * 2.0 / 3.0
    ret += (20.0 * math.sin(lng * pi) + 40.0 *
            math.sin(lng / 3.0 * pi)) * 2.0 / 3.0
    ret += (150.0 * math.sin(lng / 12.0 * pi) + 300.0 *
            math.sin(lng / 30.0 * pi)) * 2.0 / 3.0
    return ret

'''判断给定的经纬度是否位于中国境内。如果位于中国境外，返回原坐标，不进行转换。'''

def out_of_china(lng, lat):
    """
    判断是否在国内，不在国内不做偏移
    :param lng: 经度
    :param lat: 纬度
    :return: 如果坐标在中国境外，返回 True
    """
    if lng < 72.004 or lng > 137.8347:
        return True
    if lat < 0.8293 or lat > 55.8271:
        return True
    return False


poi = pd.read_excel(r'C:\Users\11707\Desktop\vspython\高德周边.xlsx', 'Sheet1')['location'].str.split(',')
poi_wgs84 = []
for p in poi:
    p_wgs84 = gcj02towgs84(float(p[0]), float(p[1]))
    poi_wgs84.append(p_wgs84)

poi_wgs84


'''
通过 pandas 读取 Excel 文件，并提取 location 列。
location 中的坐标是以逗号分隔的字符串（例如 "113.35,23.09"），通过 str.split(',') 将字符串分割成经度和纬度的列表。
然后，将每个坐标点从 GCJ-02 坐标系转换为 WGS-84 坐标系，结果存储在 poi_wgs84 列表中。
'''


'\n通过 pandas 读取 Excel 文件，并提取 location 列。\nlocation 中的坐标是以逗号分隔的字符串（例如 "113.35,23.09"），通过 str.split(\',\') 将字符串分割成经度和纬度的列表。\n然后，将每个坐标点从 GCJ-02 坐标系转换为 WGS-84 坐标系，结果存储在 poi_wgs84 列表中。\n'

In [7]:
print(poi_wgs84)

[[113.35106575231886, 23.092681452860415], [113.33781882883305, 23.09386475761564], [113.35156291724374, 23.092722644339382], [113.35163782064012, 23.09230969843387], [113.35174069638188, 23.0916228231651], [113.35002684736587, 23.087767184290982], [113.35018061923557, 23.087271143914453], [113.35021559388487, 23.086771297486735], [113.34984924649667, 23.08636606113389], [113.35285996238026, 23.08999674810146], [113.35293990541702, 23.088921075023073], [113.35324137747072, 23.089465379001926], [113.35432658695277, 23.09011444130144], [113.35348908202576, 23.087916646499313], [113.3380809311821, 23.084904039727093], [113.35169118039188, 23.100371250384796], [113.33486511985721, 23.08663622252383], [113.34114181773342, 23.10321987425094], [113.35457656144757, 23.099290228160353], [113.35468638886711, 23.099267071495724], [113.33501868607269, 23.103206016497193], [113.32991526147894, 23.098317618950116], [113.32753462758808, 23.090768295681787], [113.33753343456337, 23.107780494529155], [

In [8]:
'''替换原有坐标'''
# 将坐标对转换为逗号连接的字符串
coordinates = [f"{lng},{lat}" for lng, lat in poi_wgs84]

# 读取现有 Excel 文件
poi_all = pd.read_excel(r'C:\Users\11707\Desktop\vspython\高德周边.xlsx', 'Sheet1')

# 添加新列
poi_all['newlocation'] = coordinates

# 删除旧列
poi_all.drop(columns=['location'], inplace=True)

# 保存回 Excel 文件
poi_all.to_excel(r'C:\Users\11707\Desktop\vspython\高德周边.xlsx', sheet_name='Sheet1')
print("坐标数据已添加为 Excel 文件的新列")

坐标数据已添加为 Excel 文件的新列
